# hipinfer on Colab

LLM inference engine written from scratch in HIP (FlashAttention-style attention, tensor-core GEMM,
fused GEMV/RMSNorm/RoPE kernels), built here with `nvcc` for the attached NVIDIA GPU.

**Runtime → Change runtime type → a GPU** (T4, L4, A100 or G4). The CPU and TPU runtimes cannot run
it: the kernels are GPU code (CUDA on NVIDIA, ROCm on AMD).

Run all cells. Per model the run takes ~10 min (kernel tests once, then HF token gate, WikiText-2 perplexity
vs HF, and benchmarks; most of the time is the HF reference). `MODELS` below picks the models; the defaults
fit a 15 GB T4. On an L4 (24 GB) add `llama32-3b`; on an A100 add `mistral-7b llama31-8b` too. It ends by
downloading `hipinfer_results_<gpu>.zip`; send that back to compare GPUs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -2
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| capability", torch.cuda.get_device_capability())

Unpack the sources (embedded below so no clone is needed).

In [ ]:
import base64, io, tarfile, os
BLOB = "..."
os.makedirs("/content/hipinfer", exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(BLOB)), mode="r:gz").extractall("/content/hipinfer")
%cd /content/hipinfer
!ls -R src scripts tests

Build, kernel tests (vs PyTorch), then per model: HF token gate, WikiText-2 perplexity vs HF, benchmarks.
`GEMM_SWEEP=1` also times every GEMM tile config for the first model (adds ~3 min). Weights are exported
from the HF checkpoints on first use (`unsloth/Llama-3.2-*` are ungated mirrors of Meta's weights).

In [ ]:
!pip install -q datasets nvidia-ml-py
import torch
gpu = torch.cuda.get_device_name().replace(" ", "_")
MODELS = "tinyllama llama32-1b qwen25-0.5b qwen25-1.5b"   # + llama32-3b (24 GB GPUs), mistral-7b llama31-8b (40 GB+)
!chmod +x scripts/run_all.sh
!PY=python GEMM_SWEEP=1 PPL_WINDOWS=8 scripts/run_all.sh colab_{gpu} {MODELS}

Summary tables (the same script produces results/SUMMARY.md in the repo).

In [ ]:
!python scripts/summarize.py results

In [ ]:
!zip -qr hipinfer_results_{gpu}.zip results
from google.colab import files
files.download(f"hipinfer_results_{gpu}.zip")